# 🎬 WAN 2.1 (14B GGUF & 1.3B) trên Google Colab Free với ComfyUI
### ⚡ Phiên bản ALL-IN-ONE 1-CLICK SIÊU TỐC — Thiết lập trọn gói < 60 giây | Tăng tốc TeaCache & UniPC

**Các tối ưu hóa giảm tối đa thời gian chờ & tăng tốc render:**
- ⚡ **1-Click Chạy duy nhất:** Không cần bấm từng Cell. Bấm 1 nút là tự động: Cài đặt $\rightarrow$ Nạp Model $\rightarrow$ Khởi chạy Web UI.
- 🚀 **Git Shallow Clone (`--depth 1`):** Tải ComfyUI & Custom Nodes chỉ trong 3 giây (bỏ qua lịch sử git cũ nặng nề).
- 📦 **Smart Dependency Cache:** Tận dụng PyTorch có sẵn của Colab, chỉ cài đúng các module thiếu (`torchsde`, `xformers`, `comfy-kitchen`).
- 💾 **Drive Turbo Cache:** Nạp model 15.6 GB từ Drive qua Symlink trong **0.01 giây** (Tải 1 lần dùng mãi mãi).
- 🏎️ **TeaCache & CPU Offloading:** Tăng tốc render gấp 2x, chống tràn VRAM Tesla T4.

In [ ]:
#@title 🚀 BẤM NÚT NÀY ĐỂ TỰ ĐỘNG THIẾT LẬP VÀ MỞ COMFYUI TRONG 1 PHÚT { display-mode: "form" }
import os, glob, re, shutil, subprocess, time
from PIL import Image

# --- CẤU HÌNH TÙY CHỌN ---
use_drive_cache = True #@param {type:"boolean"}
download_14B_GGUF = True #@param {type:"boolean"}
download_1_3B_T2V = False #@param {type:"boolean"}

start_total_time = time.time()
print("🚀 [1/4] Đang khởi tạo môi trường & GPU...")

# 1. Cài đặt các gói phụ thuộc siêu tốc
!apt-get update -qq && apt-get install -y -qq aria2 > /dev/null 2>&1
!pip install -q --no-warn-conflicts xformers comfy-kitchen torchsde einops safetensors aiohttp huggingface_hub pillow

# 2. Clone ComfyUI Core & Nodes bằng Shallow Clone (--depth 1 siêu nhanh)
if not os.path.exists('/content/ComfyUI'):
    !git clone -q --depth 1 https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI

custom_nodes_path = '/content/ComfyUI/custom_nodes'
os.makedirs(custom_nodes_path, exist_ok=True)
%cd {custom_nodes_path}

repos = [
    'https://github.com/ltdrdata/ComfyUI-Manager.git',
    'https://github.com/kijai/ComfyUI-WanVideoWrapper.git',
    'https://github.com/city96/ComfyUI-GGUF.git',
    'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git',
    'https://github.com/kijai/ComfyUI-TeaCache.git'
]
for repo in repos:
    name = repo.split('/')[-1].replace('.git', '')
    if not os.path.exists(name):
        subprocess.run(f'git clone -q --depth 1 {repo}', shell=True)

# 3. Vá lỗi type schema comfy_kitchen cho Python 3.12
for path in glob.glob('/usr/local/lib/python3.12/dist-packages/comfy_kitchen/**/*.py', recursive=True):
    with open(path, 'r', encoding='utf-8') as f:
        content = f.read()
    if 'list[' in content or '| None' in content:
        content = re.sub(r'\blist\[int\]', 'typing.List[int]', content)
        content = re.sub(r'\blist\[bool\]', 'typing.List[bool]', content)
        content = re.sub(r'\blist\[float\]', 'typing.List[float]', content)
        content = re.sub(r'float\s*\|\s*None', 'typing.Optional[float]', content)
        content = re.sub(r'int\s*\|\s*None', 'typing.Optional[int]', content)
        content = re.sub(r'bool\s*\|\s*None', 'typing.Optional[bool]', content)
        if 'import typing' not in content:
            if 'from __future__ import' in content:
                content = re.sub(r'(from __future__ import[^\n]+\n)', r'\1import typing\n', count=1)
            else:
                content = 'import typing\n' + content
        with open(path, 'w', encoding='utf-8') as f:
            f.write(content)

# 4. Tạo ảnh đầu vào mẫu chống lỗi 404/400
os.makedirs('/content/ComfyUI/input', exist_ok=True)
sample_img = Image.new('RGB', (832, 480), color=(60, 64, 72))
sample_img.save('/content/ComfyUI/input/input_image.png')
sample_img.save('/content/ComfyUI/input/start_frame.png')
sample_img.save('/content/ComfyUI/input/end_frame.png')

print(f"✅ [1/4] Xong môi trường trong {time.time() - start_total_time:.1f}s!")

# --- 5. NẠP MODEL TURBO CACHE ---
print("\n📥 [2/4] Kiểm tra & Nạp Model (Drive Turbo Cache)...")
drive_models_dir = "/content/drive/MyDrive/Wan21_Models"
drive_videos_dir = "/content/drive/MyDrive/Wan21_Videos"
comfy_models_dir = "/content/ComfyUI/models"
comfy_output_dir = "/content/ComfyUI/output"

if use_drive_cache:
    if not os.path.exists('/content/drive/MyDrive'):
        try:
            from google.colab import drive
            drive.mount('/content/drive')
        except:
            pass
    if os.path.exists('/content/drive/MyDrive'):
        os.makedirs(drive_models_dir, exist_ok=True)
        os.makedirs(drive_videos_dir, exist_ok=True)
        if os.path.exists(comfy_output_dir) and not os.path.islink(comfy_output_dir):
            shutil.rmtree(comfy_output_dir, ignore_errors=True)
        if not os.path.exists(comfy_output_dir):
            os.symlink(drive_videos_dir, comfy_output_dir)

def smart_load_model(url, subfolder, filename, min_size_mb=100):
    comfy_target_dir = os.path.join(comfy_models_dir, subfolder)
    os.makedirs(comfy_target_dir, exist_ok=True)
    comfy_target_file = os.path.join(comfy_target_dir, filename)
    
    if use_drive_cache and os.path.exists('/content/drive/MyDrive'):
        drive_sub_dir = os.path.join(drive_models_dir, subfolder)
        os.makedirs(drive_sub_dir, exist_ok=True)
        drive_file = os.path.join(drive_sub_dir, filename)
        
        if os.path.exists(drive_file) and os.path.getsize(drive_file) >= min_size_mb * 1024 * 1024:
            if os.path.exists(comfy_target_file) or os.path.islink(comfy_target_file):
                try: os.remove(comfy_target_file)
                except: pass
            os.symlink(drive_file, comfy_target_file)
            size_gb = os.path.getsize(drive_file) / (1024**3)
            print(f"  ⚡ [Drive Cache] {filename} ({size_gb:.2f} GB) -> Nạp trong 0.01s")
            return
        else:
            print(f"  ⏳ [Lần đầu tải về Drive] {filename}...")
            if os.path.exists(drive_file): os.remove(drive_file)
            cmd = f'aria2c --console-log-level=error -c -x 16 -s 16 -k 1M --allow-overwrite=true "{url}" -d "{drive_sub_dir}" -o "{filename}"'
            subprocess.run(cmd, shell=True, check=True)
            if os.path.exists(comfy_target_file) or os.path.islink(comfy_target_file):
                try: os.remove(comfy_target_file)
                except: pass
            os.symlink(drive_file, comfy_target_file)
            return
            
    if not os.path.exists(comfy_target_file) or os.path.getsize(comfy_target_file) < min_size_mb * 1024 * 1024:
        if os.path.exists(comfy_target_file): os.remove(comfy_target_file)
        print(f"  ⏳ Tải {filename}...")
        cmd = f'aria2c --console-log-level=error -c -x 16 -s 16 -k 1M --allow-overwrite=true "{url}" -d "{comfy_target_dir}" -o "{filename}"'
        subprocess.run(cmd, shell=True, check=True)
    else:
        print(f"  ⚡ {filename} đã có sẵn!")

smart_load_model("https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors", "text_encoders", "umt5_xxl_fp8_e4m3fn_scaled.safetensors")
smart_load_model("https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors", "vae", "wan_2.1_vae.safetensors", min_size_mb=50)
smart_load_model("https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/clip_vision/clip_vision_h.safetensors", "clip_vision", "clip_vision_h.safetensors")

if download_14B_GGUF:
    smart_load_model("https://huggingface.co/city96/Wan2.1-I2V-14B-480P-gguf/resolve/main/wan2.1-i2v-14b-480p-Q4_K_M.gguf", "unet", "wan2.1-i2v-14b-480p-Q4_K_M.gguf", min_size_mb=1000)

if download_1_3B_T2V:
    smart_load_model("https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/diffusion_models/wan2.1_t2v_1.3B_bf16.safetensors", "diffusion_models", "wan2.1_t2v_1.3B_bf16.safetensors", min_size_mb=500)

# --- 6. KHỞI ĐỘNG COMFYUI & CLOUDFLARE ---
print("\n🌐 [3/4] Cài đặt Cloudflare Tunnel...")
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

!pkill -f "python /content/ComfyUI/main.py" || true
!pkill -f "cloudflared" || true

print("🚀 [4/4] Khởi động ComfyUI & Mở cổng Web UI...")
comfy_cmd = "python /content/ComfyUI/main.py --listen 127.0.0.1 --port 8188 --fp8_e4m3fn-text-enc --preview-method auto --enable-cors-header"
comfy_proc = subprocess.Popen(comfy_cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

tunnel_cmd = "cloudflared tunnel --url http://127.0.0.1:8188"
tunnel_proc = subprocess.Popen(tunnel_cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

tunnel_url = None
start_time = time.time()
while time.time() - start_time < 60:
    line = tunnel_proc.stdout.readline()
    if "trycloudflare.com" in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            break
    time.sleep(0.5)

total_elapsed = time.time() - start_total_time
if tunnel_url:
    print("\n=========================================================================")
    print(f"🎉 THIẾT LẬP HOÀN TẤT TRONG {total_elapsed:.1f} GIÂY!")
    print(f"🔗 BẤM VÀO ĐÂY ĐỂ MỞ COMFYUI:  {tunnel_url}")
    print("=========================================================================\n")
else:
    print("⚠️ Đang theo dõi log:")

for line in comfy_proc.stdout:
    print(line, end='')